In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import torch
from pytorch_implementation.petr.config import debug_forward_config
from pytorch_implementation.petr.model import PETRLite

cfg = debug_forward_config(num_queries=48, decoder_layers=2, depth_num=6)
model = PETRLite(cfg).eval()

batch_size = 1
height, width = 96, 160
img = torch.randn(batch_size, cfg.num_cams, 3, height, width)

def _build_dummy_img_metas(batch_size, num_cams, height, width):
    metas = []
    for batch_idx in range(batch_size):
        lidar2img = []
        for cam_idx in range(num_cams):
            projection = [
                [1.0, 0.0, 0.0, float(width) * (0.5 + cam_idx * 0.01)],
                [0.0, 1.0, 0.0, float(height) * 0.5],
                [0.0, 0.0, 1.0, 0.0],
                [0.0, 0.0, 0.0, 1.0],
            ]
            lidar2img.append(projection)
        metas.append({
            "sample_idx": f"sample_{batch_idx}",
            "lidar2img": lidar2img,
            "img_shape": [(height, width, 3)] * num_cams,
            "pad_shape": [(height, width, 3)] * num_cams,
        })
    return metas

img_metas = _build_dummy_img_metas(batch_size, cfg.num_cams, height, width)

print(f"Model: {type(model).__name__}")
print(f"Config: {cfg.name}")
print(f"Input image: {tuple(img.shape)}")
print(f"embed_dims={cfg.embed_dims}, num_queries={cfg.num_queries}, "
      f"decoder_layers={cfg.num_decoder_layers}, depth_num={cfg.depth_num}")

In [ ]:
from typing import Any

def _first_tensor(value: Any):
    """Extract the first tensor from a nested structure."""
    import torch
    if torch.is_tensor(value):
        return value
    if isinstance(value, (tuple, list)):
        for item in value:
            t = _first_tensor(item)
            if t is not None:
                return t
    if isinstance(value, dict):
        for item in value.values():
            t = _first_tensor(item)
            if t is not None:
                return t
    return None

def _iter_tensors(value: Any):
    """Iterate over all tensors in a nested structure."""
    import torch
    if torch.is_tensor(value):
        yield value
    elif isinstance(value, (tuple, list)):
        for item in value:
            yield from _iter_tensors(item)
    elif isinstance(value, dict):
        for item in value.values():
            yield from _iter_tensors(item)

def _register_hook(module, name: str, capture: dict, handles: list) -> None:
    """Register a forward hook that stores output in capture[name]."""
    def _hook(_module, _inputs, output):
        capture[name] = output
    handles.append(module.register_forward_hook(_hook))

def _print_shape(label: str, value) -> None:
    """Print shape of first tensor in value."""
    t = _first_tensor(value)
    if t is not None:
        print(f"  {label}: {tuple(t.shape)}")
    else:
        print(f"  {label}: <no tensor>")

def _check_finite(capture: dict, outputs: dict) -> None:
    """Assert all captured intermediates and outputs are finite."""
    import torch
    for name, value in capture.items():
        for t in _iter_tensors(value):
            assert torch.isfinite(t).all(), f"Non-finite in {name}"
    for name, value in outputs.items():
        if value is None:
            continue
        for t in _iter_tensors(value):
            assert torch.isfinite(t).all(), f"Non-finite in output {name}"
    print("All intermediate and final tensors are finite.")

# PETR Paper-to-Code Study Guide

This note maps PETR paper symbols/equations to the pure-PyTorch forward implementation in this repository.

Primary references:
- Paper: `papers/PETR.pdf`
- Implementation: `pytorch_implementation/petr/`
- Intermediate tensor tests: `tests/petr/test_intermediate_tensors.py`

## 1) Canonical study setup (fixed debug run)

Use one setup so equation-to-tensor mapping stays stable across sections.

- Config:
  - `debug_forward_config(num_queries=48, decoder_layers=2, depth_num=6)`
- Input image:
  - `img`: `[B, Ncam, C, H, W] = [1, 6, 3, 96, 160]`
- Metadata (`img_metas`):
  - `img_shape`: per-camera `(96, 160, 3)`
  - `pad_shape`: per-camera `(96, 160, 3)`
  - `lidar2img`: `6 x (4x4)` projection matrices

Core dimensions under this setup:
- `embed_dims = 256`
- `num_classes = 10`
- `num_decoder_layers = 2`
- `num_queries = 48`
- `depth_num = 6`

Expected model outputs:
- `all_cls_scores`: `[L, B, Q, num_classes] = [2, 1, 48, 10]`
- `all_bbox_preds`: `[L, B, Q, code_size] = [2, 1, 48, 10]`

These are verified in `tests/petr/test_intermediate_tensors.py`.

## 2) Symbol dictionary (paper -> code tensors)

- `F_t^i` (camera feature for camera `i`) -> `mlvl_feats[0][:, i]`
- `P_img` (image-space positional signal) -> `sin_embed`
- `P_3d` (3D-aware positional signal) -> `coords_position_embeding`
- `P` (final memory key position) -> `pos_embed = P_3d + P_img`
- `r_q` (3D reference points for queries) -> `reference_points`
- `e_q` (query embedding) -> `query_embeds = query_embedding(pos2posemb3d(reference_points))`
- `H_l` (decoder hidden at layer `l`) -> `outs_dec[l]`
- `\hat{c}_l` (class logits layer `l`) -> `outputs_classes[l]`
- `\hat{b}_l` (box prediction layer `l`) -> `outputs_coords[l]`

Equation IDs below are stable and use `E<section>.<index>`.

---

## Chunk 0 - End-to-end forward contract

### Goal
Bind PETR high-level pipeline to concrete module calls.

### Paper concept/equation
PETR performs object-query decoding from multi-view image memory with 3D position cues.

### Explicit equations
`(E0.1)` Feature extraction and decoding:

$$
F_t = \mathrm{ImageEncoder}(I_t), \quad H = \mathrm{Decoder}(Q, F_t, P), \quad \hat{Y} = \mathrm{Head}(H)
$$

`(E0.2)` Layer-wise outputs:

$$
\hat{Y} = \{(\hat{c}_l, \hat{b}_l)\}_{l=1}^{L}
$$

### Symbol table (E0.*)
- `I_t`: multi-camera image tensor
- `F_t`: multi-camera memory features
- `Q`: object queries
- `P`: positional embedding added to memory keys
- `\hat{Y}`: class and box predictions across decoder layers

### Code mapping
- `PETRLite.forward` in `pytorch_implementation/petr/model.py`
- `PETRLite.extract_img_feat` in `pytorch_implementation/petr/model.py`
- `PETRHeadLite.forward` in `pytorch_implementation/petr/head.py`

### Tensor shape notes
- Input image: `[B, Ncam, 3, H, W]`
- Head outputs: `all_cls_scores [L, B, Q, Ccls]`, `all_bbox_preds [L, B, Q, Cbox]`

### One sanity check
`tests/petr/test_intermediate_tensors.py` asserts final output shapes for the debug config.

---

In [ ]:
# Chunk 0: End-to-end forward pass
# Source: pytorch_implementation/petr/model.py - PETRLite.forward

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)

print("=== End-to-end output shapes ===")
for key, val in outputs.items():
    if val is not None and torch.is_tensor(val):
        print(f"  {key}: {tuple(val.shape)}")
    elif val is None:
        print(f"  {key}: None")

assert outputs["all_cls_scores"].shape == (cfg.num_decoder_layers, batch_size, cfg.num_queries, cfg.num_classes)
assert outputs["all_bbox_preds"].shape == (cfg.num_decoder_layers, batch_size, cfg.num_queries, cfg.code_size)
print("\nShape assertions passed.")

## Chunk 1 - Image features (backbone + neck)

### Goal
Understand how camera images are flattened, encoded, and restored to camera-major shape.

### Paper concept/equation
Each camera image is processed by shared CNN weights, then fused as a multi-view memory set.

### Explicit equations
`(E1.1)` Camera-batch flattening:

$$
I_t \in \mathbb{R}^{B\times N_{cam}\times 3\times H\times W}
\rightarrow
I'_t \in \mathbb{R}^{(B\cdot N_{cam})\times 3\times H\times W}
$$

`(E1.2)` Feature reshape back to camera axis:

$$
F'_t \in \mathbb{R}^{(B\cdot N_{cam})\times C\times H_f\times W_f}
\rightarrow
F_t \in \mathbb{R}^{B\times N_{cam}\times C\times H_f\times W_f}
$$

### Symbol table (E1.*)
- `N_cam`: number of cameras
- `H_f, W_f`: feature-map size after backbone/neck
- `C`: neck output channels

### Code mapping
- `BackboneNeck` in `pytorch_implementation/petr/backbone_neck.py`
- `extract_img_feat` in `pytorch_implementation/petr/model.py`

### Tensor shape notes
- Debug run yields `fpn.output0` shape `[6, 256, 6, 10]`
- After reshape: `[1, 6, 256, 6, 10]`

### One sanity check
`tests/petr/test_intermediate_tensors.py` validates backbone stage and `fpn.output0` shapes.

---

In [ ]:
# Chunk 1: Image features (backbone + neck)
# Source: pytorch_implementation/petr/model.py - PETRLite.extract_img_feat
# Source: pytorch_implementation/petr/backbone_neck.py - BackboneNeck

capture, handles = {}, []

_register_hook(model.backbone_neck.backbone.stem, "backbone.stem", capture, handles)
for idx, stage in enumerate(model.backbone_neck.backbone.stages):
    _register_hook(stage, f"backbone.stage{idx}", capture, handles)
_register_hook(model.backbone_neck.neck.output_convs[0], "fpn.output0", capture, handles)

with torch.no_grad():
    img_feats = model.extract_img_feat(img)

for h in handles:
    h.remove()

print("=== Backbone stage shapes (B*Ncam flattened) ===")
_print_shape("backbone.stem", capture["backbone.stem"])
for idx in range(4):
    _print_shape(f"backbone.stage{idx}", capture[f"backbone.stage{idx}"])

print("\n=== FPN output ===")
_print_shape("fpn.output0", capture["fpn.output0"])

print("\n=== Camera-major feature (after reshape) ===")
for lvl, feat in enumerate(img_feats):
    print(f"  mlvl_feats[{lvl}]: {tuple(feat.shape)}")
    assert feat.shape[0] == batch_size
    assert feat.shape[1] == cfg.num_cams

## Chunk 2 - 3D position embedding from geometry

### Goal
Connect PETR's 3D coordinate lifting with implemented tensor operations.

### Paper concept/equation
Image-grid points with sampled depths are lifted through inverse camera projection to 3D and normalized in a predefined range.

### Explicit equations
`(E2.1)` Pixel-depth homogeneous point:

$$
\tilde{p}(u,v,d) = [u\cdot d, v\cdot d, d, 1]^T
$$

`(E2.2)` Lift to 3D (lidar/world frame proxy):

$$
p_{3d} = T^{-1}_{lidar2img}\,\tilde{p}
$$

`(E2.3)` Range normalization:

$$
\bar{p}_{3d} = \frac{p_{3d} - p_{min}}{p_{max} - p_{min}}
$$

### Symbol table (E2.*)
- `(u,v)`: image coordinates at feature resolution
- `d`: sampled depth bin
- `T_{lidar2img}`: camera projection matrix from metadata
- `p_min, p_max`: `position_range` bounds

### Code mapping
- `PETRHeadLite.position_embeding` in `pytorch_implementation/petr/head.py`
- `inverse_sigmoid` in `pytorch_implementation/petr/utils.py`
- `position_encoder` in `pytorch_implementation/petr/head.py`

### Tensor shape notes
- Pre-conv geometry tensor: `[B*Ncam, 3*D, H_f, W_f]`
- Encoded 3D positional feature: `[B, Ncam, C, H_f, W_f]`

### One sanity check
`head.position_encoder` hook is asserted to output `[B*Ncam, C, H_f, W_f]`.

---

In [ ]:
# Chunk 2: 3D position embedding from geometry
# Source: pytorch_implementation/petr/head.py - PETRHeadLite.position_embeding
# Source: pytorch_implementation/petr/utils.py - inverse_sigmoid, SinePositionalEncoding2D

capture, handles = {}, []
_register_hook(model.head.input_proj, "head.input_proj", capture, handles)
_register_hook(model.head.position_encoder, "head.position_encoder", capture, handles)
_register_hook(model.head.adapt_pos3d, "head.adapt_pos3d", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)

for h in handles:
    h.remove()

print("=== 3D position embedding shapes ===")
_print_shape("head.input_proj (B*Ncam, C, Hf, Wf)", capture["head.input_proj"])
_print_shape("head.position_encoder (B*Ncam, C, Hf, Wf)", capture["head.position_encoder"])
_print_shape("head.adapt_pos3d (B*Ncam, C, Hf, Wf)", capture["head.adapt_pos3d"])

# Replay the geometry step standalone
x = img_feats[0]
import torch.nn.functional as F
masks = model.head._build_img_masks(img_metas, batch_size=batch_size,
                                     num_cams=cfg.num_cams, device=x.device)
x_proj = model.head.input_proj(x.flatten(0, 1)).view(
    batch_size, cfg.num_cams, cfg.embed_dims, x.shape[-2], x.shape[-1])
masks_resized = F.interpolate(masks.float(), size=x_proj.shape[-2:], mode="nearest").to(torch.bool)

coords_pos, coords_mask = model.head.position_embeding(x_proj, img_metas, masks_resized)
print(f"\n=== Standalone geometry replay ===")
print(f"  coords_position_embedding (P_3d): {tuple(coords_pos.shape)}")
print(f"  coords_mask: {tuple(coords_mask.shape)}")

## Chunk 3 - Query construction from 3D reference points

### Goal
Map learned reference anchors to decoder query embeddings.

### Paper concept/equation
PETR uses learnable 3D reference points and sinusoidal embedding to parameterize object queries.

### Explicit equations
`(E3.1)` Learnable reference points:

$$
r_q \in \mathbb{R}^{Q\times 3}
$$

`(E3.2)` Query embedding:

$$
e_q = \mathrm{MLP}(\mathrm{PE}_{3d}(r_q))
$$

### Symbol table (E3.*)
- `Q`: number of object queries
- `r_q`: normalized 3D reference points
- `\mathrm{PE}_{3d}`: sinusoidal embedding (`pos2posemb3d`)
- `e_q`: decoder query positional embedding

### Code mapping
- `reference_points` and `query_embedding` in `pytorch_implementation/petr/head.py`
- `pos2posemb3d` in `pytorch_implementation/petr/utils.py`

### Tensor shape notes
- `reference_points.weight`: `[Q, 3]`
- `query_embeds`: `[Q, C]`

### One sanity check
Tests assert `head.reference_points` and `head.query_embedding` output shapes.

---

In [ ]:
# Chunk 3: Query construction from 3D reference points
# Source: pytorch_implementation/petr/head.py - reference_points, query_embedding
# Source: pytorch_implementation/petr/utils.py - pos2posemb3d

from pytorch_implementation.petr.utils import pos2posemb3d

capture, handles = {}, []
_register_hook(model.head.reference_points, "head.reference_points", capture, handles)
_register_hook(model.head.query_embedding, "head.query_embedding", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)

for h in handles:
    h.remove()

print("=== Query construction shapes ===")
_print_shape("reference_points (r_q): [Q, 3]", capture["head.reference_points"])
_print_shape("query_embedding (e_q): [Q, C]", capture["head.query_embedding"])

# Replay standalone: show the pos2posemb3d step
ref_pts = model.head.reference_points(torch.arange(cfg.num_queries, device=img.device))
print(f"\n=== Standalone query replay ===")
print(f"  raw reference_points (learned, before sigmoid): {tuple(ref_pts.shape)}")
pos_emb_3d = pos2posemb3d(ref_pts, num_pos_feats=cfg.embed_dims // 2)
print(f"  pos2posemb3d output: {tuple(pos_emb_3d.shape)}")
query_embeds = model.head.query_embedding(pos_emb_3d)
print(f"  query_embedding output: {tuple(query_embeds.shape)}")

## Chunk 4 - Transformer decoder over multi-view memory

### Goal
Describe how flattened memory and query tokens interact through self/cross attention.

### Paper concept/equation
PETR decodes object tokens using decoder layers: self-attention, cross-attention to image memory, and FFN refinement.

### Explicit equations
`(E4.1)` Memory flattening:

$$
M \in \mathbb{R}^{(N_{cam}H_fW_f)\times B\times C}
$$

`(E4.2)` Decoder layer update:

$$
H_l = \mathrm{FFN}(\mathrm{CrossAttn}(\mathrm{SelfAttn}(H_{l-1})))
$$

### Symbol table (E4.*)
- `M`: flattened memory tokens from camera features
- `H_l`: decoder hidden state at layer `l`
- `L`: number of decoder layers

### Code mapping
- `PETRTransformerLite.forward` in `pytorch_implementation/petr/transformer.py`
- `PETRTransformerDecoderLayerLite` in `pytorch_implementation/petr/transformer.py`

### Tensor shape notes
- Decoder hidden per layer: `[Q, B, C]`
- Stacked decoder output: `[L, Q, B, C]`

### One sanity check
Tests verify each `decoder.layer*`, `self_attn`, `cross_attn`, and `ffn` output has shape `[Q, B, C]`.

---

In [ ]:
# Chunk 4: Transformer decoder over multi-view memory
# Source: pytorch_implementation/petr/transformer.py
#   PETRTransformerLite.forward, PETRTransformerDecoderLayerLite.forward

capture, handles = {}, []
for idx, layer in enumerate(model.head.transformer.decoder.layers):
    _register_hook(layer, f"decoder.layer{idx}", capture, handles)
    _register_hook(layer.self_attn, f"decoder.layer{idx}.self_attn", capture, handles)
    _register_hook(layer.cross_attn, f"decoder.layer{idx}.cross_attn", capture, handles)
    _register_hook(layer.ffn, f"decoder.layer{idx}.ffn", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)

for h in handles:
    h.remove()

print("=== Decoder layer shapes ===")
for idx in range(cfg.num_decoder_layers):
    print(f"\n  --- Layer {idx} ---")
    _print_shape(f"  self_attn  [Q, B, C]", capture[f"decoder.layer{idx}.self_attn"])
    _print_shape(f"  cross_attn [Q, B, C]", capture[f"decoder.layer{idx}.cross_attn"])
    _print_shape(f"  ffn        [Q, B, C]", capture[f"decoder.layer{idx}.ffn"])
    _print_shape(f"  layer_out  [Q, B, C]", capture[f"decoder.layer{idx}"])

    t = _first_tensor(capture[f"decoder.layer{idx}"])
    assert t.shape == (cfg.num_queries, batch_size, cfg.embed_dims), \
        f"Unexpected decoder.layer{idx} shape: {tuple(t.shape)}"

print("\nAll decoder layer shape assertions passed.")

## Chunk 5 - Class/box heads and metric-space decoding

### Goal
Map decoder states to class logits and 3D box predictions.

### Paper concept/equation
Each decoder layer predicts class scores and box parameters; center dimensions tied to reference points use inverse-sigmoid residual updates.

### Explicit equations
`(E5.1)` Per-layer predictions:

$$
\hat{c}_l = f_{cls}(H_l), \quad \hat{b}_l = f_{reg}(H_l)
$$

`(E5.2)` Reference-aware center update:

$$
\hat{x},\hat{y}=\sigma(\Delta_{xy}+\sigma^{-1}(r_{xy})), \quad
\hat{z}=\sigma(\Delta_{z}+\sigma^{-1}(r_{z}))
$$

`(E5.3)` Scale normalized centers to metric range:

$$
x = \hat{x}(x_{max}-x_{min}) + x_{min}
$$

and similarly for `y, z`.

### Symbol table (E5.*)
- `f_cls, f_reg`: layer-specific MLP heads
- `r_xy, r_z`: reference point components
- `\Delta_{xy}, \Delta_z`: residual outputs from regression branch

### Code mapping
- Layer branches in `PETRHeadLite._build_branches`
- Forward update equations in `PETRHeadLite.forward`
- Top-k decode in `NMSFreeCoderLite` (`pytorch_implementation/petr/postprocess.py`)

### Tensor shape notes
- `all_cls_scores`: `[L, B, Q, num_classes]`
- `all_bbox_preds`: `[L, B, Q, code_size]`
- Decoded inference set uses the last layer (`[-1]`) and top-k over `Q * num_classes`.

### One sanity check
Tests assert class/box branch output dimensions for every decoder layer and finite values for all captured outputs.

---

In [ ]:
# Chunk 5: Class/box heads and metric-space decoding
# Source: pytorch_implementation/petr/head.py - PETRHeadLite._build_branches, forward
# Source: pytorch_implementation/petr/postprocess.py - NMSFreeCoderLite

capture, handles = {}, []
for idx, branch in enumerate(model.head.cls_branches):
    _register_hook(branch, f"head.cls_branch{idx}", capture, handles)
for idx, branch in enumerate(model.head.reg_branches):
    _register_hook(branch, f"head.reg_branch{idx}", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)

for h in handles:
    h.remove()

print("=== Per-layer branch output shapes ===")
for idx in range(cfg.num_decoder_layers):
    _print_shape(f"  cls_branch{idx} [B, Q, num_classes]", capture[f"head.cls_branch{idx}"])
    _print_shape(f"  reg_branch{idx} [B, Q, code_size]", capture[f"head.reg_branch{idx}"])

print("\n=== Stacked outputs ===")
print(f"  all_cls_scores: {tuple(outputs['all_cls_scores'].shape)}")
print(f"  all_bbox_preds: {tuple(outputs['all_bbox_preds'].shape)}")

# Run decode (NMS-free top-k)
with torch.no_grad():
    decoded_outputs = model(img, img_metas, decode=True)

decoded = decoded_outputs["decoded"]
print(f"\n=== Decoded predictions (sample 0) ===")
print(f"  bboxes: {tuple(decoded[0]['bboxes'].shape)}")
print(f"  scores: {tuple(decoded[0]['scores'].shape)}")
print(f"  labels: {tuple(decoded[0]['labels'].shape)}")
print(f"  labels dtype: {decoded[0]['labels'].dtype}")

## 3) Dataflow diagram

```mermaid
flowchart LR
    imgInput["MultiCameraImage BxNcamx3xHxW"] --> backboneNeck[BackboneNeck]
    backboneNeck --> mlvlFeats["mlvl_feats BxNcamxCxHfxWf"]
    mlvlFeats --> inputProj[input_proj]
    inputProj --> posEmbed["position_embeding + adapt_pos3d"]
    imgMetas["img_metas lidar2img"] --> posEmbed
    posEmbed --> keyPos["pos_embed = P_3d + P_img"]
    inputProj --> flattenMemory["Flatten to NcamHfWf x B x C"]
    keyPos --> decoder[TransformerDecoder]
    flattenMemory --> decoder
    refPoints["reference_points Q x 3"] --> queryEmbed["pos2posemb3d + query_embedding"]
    queryEmbed --> decoder
    decoder --> outsDec["outs_dec L x Q x B x C"]
    outsDec --> clsBranches[cls_branches]
    outsDec --> regBranches[reg_branches]
    clsBranches --> clsScores["all_cls_scores L x B x Q x Ccls"]
    regBranches --> bboxPreds["all_bbox_preds L x B x Q x Cbox"]
    clsScores --> decode[NMSFreeCoderLite]
    bboxPreds --> decode
```

## 4) One end-to-end tensor trace

1. Start with `img [1, 6, 3, 96, 160]`.
2. Backbone+FPN returns one level `[1, 6, 256, 6, 10]`.
3. `input_proj` projects to `[6, 256, 6, 10]`.
4. `position_embeding` lifts pixel-depth points to 3D and encodes: `coords_position_embedding [1, 6, 256, 6, 10]`.
5. Sine 2D positional encoding + `adapt_pos3d`: `sin_embed [1, 6, 256, 6, 10]`.
6. `pos_embed = coords_position_embedding + sin_embed`: `[1, 6, 256, 6, 10]`.
7. Flatten camera memory: `memory [360, 1, 256]` (6 cams * 6 * 10 = 360 tokens).
8. `key_pos` flattened similarly: `[360, 1, 256]`.
9. `key_padding_mask`: `[1, 360]`.
10. Reference points: `[48, 3]` -> `pos2posemb3d` -> `query_embeds [48, 256]`.
11. Query initialization: `target = zeros [48, 1, 256]`, `query_pos [48, 1, 256]`.
12. Run 2 decoder layers (self-attn -> cross-attn -> FFN):
    - each layer output `[48, 1, 256]`.
13. Stacked intermediate: `outs_dec [2, 48, 1, 256]` -> permuted to `[2, 1, 48, 256]`.
14. Per-layer cls/reg branches with reference-aware updates:
    - `all_cls_scores [2, 1, 48, 10]`
    - `all_bbox_preds [2, 1, 48, 10]`.
15. NMS-free decode selects top-k candidates and outputs final boxes/scores/labels.

## 5) Study drills (self-check questions)

1. Why does PETR use depth bins to lift 2D image points to 3D instead of estimating explicit depth?
2. What concrete tensors correspond to paper symbols `F_t`, `P_3d`, and `r_q`?
3. How does `position_embeding` decide which 3D points are out of range, and what mask does it produce?
4. Why are query embeddings derived from 3D reference points via sinusoidal encoding rather than being fully learned?
5. What changes in the `coords_position_embedding` tensor if you double `depth_num`?
6. Where exactly does `pos_embed = P_3d + P_img` happen in the code?
7. Why does the decoder use `key_padding_mask` based on image/pad shapes?
8. Which box coordinates use the reference-aware sigmoid residual update (`inverse_sigmoid`) vs. direct regression?
9. What is the role of `adapt_pos3d` — why not use `sin_embed` directly?
10. If all cameras had identical `lidar2img` matrices, what symmetry would you see in the 3D position embeddings?

## 6) Practical reading order for this note

1. Read Sections 1 and 2 once.
2. Walk through Chunk 1 (backbone) then Chunk 2 (3D position embedding) to understand inputs.
3. Study Chunk 3 (query construction from reference points).
4. Study Chunk 4 (decoder layers).
5. Study Chunk 5 (heads and decode).
6. Re-read Chunk 0 (end-to-end) to tie the full pipeline together.
7. Re-run the end-to-end trace in Section 4 while stepping through code.
8. Answer study drills without looking at code, then verify.

## 7) Known implementation simplifications in this repo

- Uses standard `nn.MultiheadAttention` instead of deformable attention.
- Single FPN level only (no multi-scale features).
- No data augmentation transforms in the geometry pipeline.
- `lidar_discretization` depth binning is configurable but defaults to uniform spacing.

These simplifications keep the PETR concept flow explicit for study.

In [ ]:
# Final validation: all intermediate and output tensors are finite

capture, handles = {}, []
_register_hook(model.backbone_neck.backbone.stem, "backbone.stem", capture, handles)
for idx, stage in enumerate(model.backbone_neck.backbone.stages):
    _register_hook(stage, f"backbone.stage{idx}", capture, handles)
_register_hook(model.backbone_neck.neck.output_convs[0], "fpn.output0", capture, handles)
_register_hook(model.head.input_proj, "head.input_proj", capture, handles)
_register_hook(model.head.position_encoder, "head.position_encoder", capture, handles)
_register_hook(model.head.adapt_pos3d, "head.adapt_pos3d", capture, handles)
_register_hook(model.head.reference_points, "head.reference_points", capture, handles)
_register_hook(model.head.query_embedding, "head.query_embedding", capture, handles)
for idx, layer in enumerate(model.head.transformer.decoder.layers):
    _register_hook(layer, f"decoder.layer{idx}", capture, handles)
    _register_hook(layer.self_attn, f"decoder.layer{idx}.self_attn", capture, handles)
    _register_hook(layer.cross_attn, f"decoder.layer{idx}.cross_attn", capture, handles)
    _register_hook(layer.ffn, f"decoder.layer{idx}.ffn", capture, handles)
for idx, branch in enumerate(model.head.cls_branches):
    _register_hook(branch, f"head.cls_branch{idx}", capture, handles)
for idx, branch in enumerate(model.head.reg_branches):
    _register_hook(branch, f"head.reg_branch{idx}", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()

_check_finite(capture, outputs)